# Preprocessing for the Context-Aware Toxicity 
## NOT DONE SPELLCHECK AND REMOVE   "somebody said" PHRASES FROM CONTEXT

**Goal of this notebook:** take the raw `train.csv` and turn it into a clean file that any model
(TF-IDF, LSTM, BERT...) can use.

**What the notebook does, in order**

| Step | What we do | Why |
|---|---|---|
| 1 | Load the data | See what we have |
| 2 | Fill the empty `toxicity_type` with `"None"` | Empty means "not toxic" – it is a real label, not missing data |
| 3 | Keep only English rows (remove Bengali etc.) | Your teacher / task needs English only |
| 4 | Remove duplicate rows | The same row twice just makes the model memorize |
| 5 | Clean the text | Make words look the same everywhere ("Don't" = "do not") |
| 6 | Turn labels into numbers | Models only understand numbers |
| 7 | Split into train / validation / test **without leakage** | So the test score is honest |
| 8 | Save the clean file | Use it in the next notebooks |

## Step 1: Load the data

In [1]:
import re
import pandas as pd
import emoji
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv("train.csv")
print("Rows and columns:", df.shape)
df.head()

Rows and columns: (19453, 5)


,row_id,previous_reply,target_reply,level_1_class,toxicity_type
0,ef2a32c7af5a0a660e0c,I shared photos from today's event.,"In this video, your body draws more attention ...",Toxic,Sexual Harassment / Objectification
1,c7a506deea76fe585973,I see no reason to change my opinion.,"Hearing this statement, it seems imagination d...",Toxic,Mockery / Ridicule
2,5ff6cb853fa2d2809023,"Someone in the discussion said, There is no ne...",Not everyone is the same; people should be tre...,Non-toxic,NaN
3,577f23b0dbafdca5c219,What's the update on today's work?,"So far, More details can be given later.",Non-toxic,NaN
4,ca4f30bb8940a17405a0,I only disagreed politely.,"I'm warning you, if you ask this question agai...",Toxic,Threat


## Step 2: Fill the empty `toxicity_type`

For non-toxic replies the `toxicity_type` cell is empty. Python reads an empty cell as "missing" (NaN).
But for us, empty **means "not toxic"**, so we write it as the word `None` – it becomes a normal class.

In [2]:
df["toxicity_type"] = df["toxicity_type"].fillna("None")
df["toxicity_type"].value_counts()

toxicity_type
None                                   9861
Insult                                 1923
Threat                                  902
Mockery / Ridicule                      845
Hate Speech                             808
Incitement / Encouragement of Harm      800
Identity Attack                         800
Dehumanization                          796
Profanity                               779
Sexual Harassment / Objectification     763
Toxic Endorsement / Agreement           621
Harassment                              555
Name: count, dtype: int64

## Step 3: Keep only English rows

You asked to keep only rows where **both** `previous_reply` and `target_reply` are English.
A row is removed if **either** column has:

1. **Bengali letters** (they live in the Unicode range `\u0980` to `\u09FF`), or
2. **any other non-English alphabet** (Hindi, Arabic, ...), or
3. **Bengali written in English letters** ("Banglish", like *"tumi kemon acho"*). We check this with a small list of
   Bengali words. We remove the row only if it has **2 or more** of these words, so a normal English sentence is never removed by mistake.

> Why not use a language-detection library? On short sentences they make mistakes. For example, they called
> *"Bastard, dog licking Hasina's feet."* Swedish. That would delete a good English row.

In [3]:
bengali_letters = re.compile(r"[\u0980-\u09FF]")

banglish_words = set("""tumi ami tui tora amra apni amar tomar kemon keno kothay bhalo valo
nai hobe korbo korchi ekta ekhon eta ota lagbe bolo naki tahole shala shuor kutta
bokachoda pagol gadha harami tomake amake onek kichu jodi""".split())

def is_not_english(text):
    # 1) Bengali letters
    if bengali_letters.search(text):
        return True
    # 2) any letter outside the Latin alphabet (code > 591 means "not a Latin letter")
    if any(ch.isalpha() and ord(ch) > 591 for ch in text):
        return True
    # 3) romanized Bengali: 2 or more Bengali words
    words = re.findall(r"[a-z]+", text.lower())
    if sum(w in banglish_words for w in words) >= 2:
        return True
    return False

bad_prev   = df["previous_reply"].apply(is_not_english)
bad_target = df["target_reply"].apply(is_not_english)
remove = bad_prev | bad_target          # remove the row if EITHER column is not English

print("Rows with non-English text:", remove.sum())
df[remove].to_csv("removed_non_english_rows.csv", index=False)   # keep a copy to check what was removed
df = df[~remove].reset_index(drop=True)
print("Rows left:", len(df))

Rows with non-English text: 0
Rows left: 19453


**Result on your file:** 0 rows were removed. Your data has no Bengali letters and no Banglish, so it is already English.
The step is still here so the notebook is safe if you add new data later.

## Step 4: Remove duplicate rows

If the exact same (`previous_reply`, `target_reply`) pair appears twice, we keep only one copy.
Two identical rows add no new information and can trick the model into just memorizing.

> Important: we do **not** remove rows that only share the same `target_reply`. Their `previous_reply` is different,
> and the whole point of a *context-aware* model is that the same sentence can get a different label in a different context.

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["previous_reply", "target_reply"]).reset_index(drop=True)    ## Only drop hen the values in both the columns are same
print("Duplicate rows removed:", before - len(df))
print("Rows left:", len(df))

Duplicate rows removed: 22
Rows left: 19431


## Step 5: Clean the text

We make a **second copy** of each text column (`prev_clean`, `target_clean`) and leave the original untouched.

| Cleaning step | Example | Why |
|---|---|---|
| Replace fancy dashes `—` with a space | `great—wow` → `great wow` | Otherwise two words stick together |
| Turn emojis into words | 😂 → `face with tears of joy` | Emojis carry meaning (mockery!), models can read words but not emoji pictures |
| Make everything lowercase | `You` = `you` | The model should not treat them as 2 different words |
| Shorten stretched letters | `sooooo` → `soo` | Same word, different spelling |
| Expand short forms | `don't` → `do not`, `you're` → `you are` | `don't` and `do not` should be the same thing |
| Keep `!` and `?` as separate tokens | `crime!` → `crime !` | They show emotion and sarcasm |
| Remove other punctuation | `,` `.` `;` `"` | They add noise |
| Remove extra spaces | | Tidy |

**What we do NOT do:** we do **not** remove stop-words like *you, not, no, yes, right*.
Normally people delete them, but here they are the most important words
("**You** are stupid", "**Yes**, you are **right**", "I do **not** agree").

> For BERT-type models you can skip this and give them the **original** columns instead
> (they work better with the original punctuation and capital letters).

In [5]:
import re
import emoji
# from spellchecker import SpellChecker

# Initialize the spell checker once outside the function for speed
# spell = SpellChecker()

short_forms = {
    "won't": "will not", "can't": "can not", "cannot": "can not", "ain't": "is not",
    "let's": "let us", "that's": "that is", "it's": "it is", "he's": "he is", "she's": "she is",
    "there's": "there is", "what's": "what is", "who's": "who is",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am", "'d": " would",
}

def clean_text_with_spellcheck(text):
    text = str(text)
    text = text.replace("\u2014", " ").replace("\u2013", " ")       # long dashes -> space
    text = text.replace("\u2019", "'").replace("\u2018", "'")       # curly apostrophes -> normal '
    
    # emoji -> words
    text = emoji.demojize(text, delimiters=(" ", " ")) 
    text = text.lower()
    
    # 1. FIXED REGEX: Collapse 3+ repeated characters down to 2 ONLY for letters (protects numbers like 1000)
    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)               # sooooo -> soo
    
    # 2. Expand short forms
    for short, full in short_forms.items():                          # don't -> do not
        text = text.replace(short, full)
    
    text = text.replace("'s", "")                                    # remove possessive: hasina's -> hasina
    
    # 7. Clean punctuation while retaining ! and ?
    text = re.sub(r"[^a-z0-9!?_ ]", " ", text)
    text = re.sub(r"([!?])", r" \1 ", text)

    # 8. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Apply the updated function to your dataframe
# df["prev_clean"]   = df["previous_reply"].apply(clean_text_with_spellcheck)
# df["target_clean"] = df["target_reply"].apply(clean_text_with_spellcheck)

# # Look at a few examples: before and after
# df[["target_reply", "target_clean"]].sample(5, random_state=1)


In [6]:
# Quick test of the function on a made-up sentence
clean_text_with_spellcheck("You're sooooo smart, aren't you?! 😂 Great—wow!!!")

'you are soo smart are not you ? ! face_with_tears_of_joy great wow ! ! !'

In [7]:

import re

# Remove only an introductory reporting phrase.
# Preserve the content that follows it.
ATTRIBUTION_PATTERNS = [
  r"^\s*someone in the discussion said\s*[,:\-–—]?\s*",
  r"^\s*a comment said\s*[,:\-–—]?\s*",
  r"^\s*a friend said\s*[,:\-–—]?\s*",
  r"^\s*someone said\s*[,:\-–—]?\s*",
  r"^\s*somebody said\s*[,:\-–—]?\s*",
  r"^\s*someone wrote\s*[,:\-–—]?\s*",
  r"^\s*a comment stated\s*[,:\-–—]?\s*",
]

def remove_attribution(text):
  text = str(text).strip()

  for pattern in ATTRIBUTION_PATTERNS:
    text = re.sub(
      pattern,
      "",
      text,
      count=1,
      flags=re.IGNORECASE
    )

  return text.strip()

# Keep the original columns unchanged
df["previous_reply_debiased"] = (
  df["previous_reply"].apply(remove_attribution)
)

# Feed the transformed context to your existing cleaner
df["prev_clean"] = (
    df["previous_reply_debiased"]
      .apply(clean_text_with_spellcheck)
)


# Inspect before training
examples = df.loc[
  df["previous_reply"].str.match(
    r"^\s*(someone in the discussion said|a comment said|a friend said)",
    case=False,
    na=False
  ),
  ["previous_reply", "previous_reply_debiased", "prev_clean"]
]

print(examples.head(20).to_string(index=False))


                                                                                                 previous_reply                                                         previous_reply_debiased                                                                     prev_clean
                     Someone in the discussion said, There is no need to take men seriously in this discussion.                      There is no need to take men seriously in this discussion.                      there is no need to take men seriously in this discussion
                                                A comment said, Abusive language is increasing in the comments.                                 Abusive language is increasing in the comments.                                 abusive language is increasing in the comments
                                 A friend said, The interview did not go well; I think the opportunity is gone.                 The interview did not go well; I think the opportunity is g

In [11]:
df["target_clean"] = df["target_reply"].apply(clean_text_with_spellcheck)

## Step 6: Turn labels into numbers

Models cannot read the word "Insult". We give each class a number. `None` gets **0**.
We also make a simple yes/no column `is_toxic` (0 = not toxic, 1 = toxic), which is useful if you want a 2-step model
(first "toxic or not?", then "which type?").

In [8]:
classes = sorted(df["toxicity_type"].unique(), key=lambda c: (c != "None", c))   # "None" first
label_to_id = {name: number for number, name in enumerate(classes)}

df["label"]    = df["toxicity_type"]
df["label_id"] = df["label"].map(label_to_id)
df["is_toxic"] = (df["label"] != "None").astype(int)

print(label_to_id)
df["label"].value_counts()

{'None': 0, 'Dehumanization': 1, 'Harassment': 2, 'Hate Speech': 3, 'Identity Attack': 4, 'Incitement / Encouragement of Harm': 5, 'Insult': 6, 'Mockery / Ridicule': 7, 'Profanity': 8, 'Sexual Harassment / Objectification': 9, 'Threat': 10, 'Toxic Endorsement / Agreement': 11}


label
None                                   9854
Insult                                 1923
Threat                                  902
Mockery / Ridicule                      845
Hate Speech                             808
Identity Attack                         800
Incitement / Encouragement of Harm      798
Dehumanization                          787
Profanity                               779
Sexual Harassment / Objectification     763
Toxic Endorsement / Agreement           617
Harassment                              555
Name: count, dtype: int64

## Step 7: Split into train / validation / test (the careful way)

We need 3 parts:
- **train** – the model learns from this,
- **validation** – we use it to tune and compare models,
- **test** – we look at it **only once at the end** for the final score.

### The problem: the same `target_reply` appears many times
In your data, one target sentence can appear up to **80 times** (with different previous replies).
If we split rows randomly, the *same target sentence* lands in both train and test.
The model then just **remembers the sentence** and the score looks great, but the model has not really learned anything.
It is like giving students the exam questions as homework.

### The fix: keep all copies of the same target sentence together
1. We give every unique `target_reply` a **group number** (`group`). Same sentence = same group number.
   (We first make the sentence lowercase and strip punctuation, so tiny differences like a capital letter don't create a new group.)
2. `StratifiedGroupKFold` then splits the data so that:
   - **Group** → all rows of one group go to the **same** part (no leakage),
   - **Stratified** → every part gets about the same share of each toxicity type (so small classes like Harassment appear everywhere).

We cut the data into 10 parts: part 0 = **test** (10%), part 1 = **validation** (10%), the other 8 = **train** (80%).

In [12]:
# 1) a group number for every unique target sentence
target_key = df["target_clean"]
df["group"] = target_key.factorize()[0]    ## Assigns unique integer IDs to each unique target sentence

print("Rows:", len(df), "| unique target sentences:", df["group"].nunique())
print("Most repeated target sentence appears", df["group"].value_counts().max(), "times")

Rows: 19431 | unique target sentences: 9237
Most repeated target sentence appears 80 times


In [13]:
# 2) split into 10 parts: same group stays together, classes stay balanced
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

df["part"] = -1   ## Every row has not not been assigned to a fold yet
for part_number, (train_rows, part_rows) in enumerate(splitter.split(df, df["label_id"], df["group"])):  ## Three arguments: 1) the dataframe, 2) the labels (for stratification), 3) the groups (to keep together)
    df.loc[part_rows, "part"] = part_number

df["split"] = "train"
df.loc[df["part"] == 0, "split"] = "test"
df.loc[df["part"] == 1, "split"] = "val"

df["split"].value_counts()

split
train    15433
val       2111
test      1887
Name: count, dtype: int64

### Check that the split is clean
Two checks:
1. **No target sentence should appear in more than one split** (the numbers below must be 0).
2. **Every split should have all classes**, in similar proportions.

In [14]:
train_groups = set(df[df.split == "train"]["group"])
val_groups   = set(df[df.split == "val"]["group"])
test_groups  = set(df[df.split == "test"]["group"])

print("Shared target sentences  train & test:", len(train_groups & test_groups))
print("Shared target sentences  train & val :", len(train_groups & val_groups))
print("Shared target sentences  val   & test:", len(val_groups & test_groups))

Shared target sentences  train & test: 0
Shared target sentences  train & val : 0
Shared target sentences  val   & test: 0


In [15]:
# Share of each class in every split (in %)
(pd.crosstab(df["label"], df["split"], normalize="columns") * 100).round(1)

split,test,train,val
label,,,
Dehumanization,3.9,4.1,3.6
Harassment,3.6,2.8,2.9
Hate Speech,2.7,4.3,4.4
Identity Attack,4.3,4.1,3.7
Incitement / Encouragement of Harm,4.4,4.1,3.7
Insult,10.4,9.8,10.2
Mockery / Ridicule,4.6,4.5,3.3
None,49.4,50.0,56.8
Profanity,4.6,4.2,1.8


## Step 8: Save the clean file
We keep the original text **and** the cleaned text, the labels, and which split each row belongs to.

In [16]:
columns_to_save = ["row_id", "previous_reply", "target_reply",   # original text (use for BERT)
                   "prev_clean", "target_clean",                 # cleaned text (use for TF-IDF / LSTM)
                   "label", "label_id", "is_toxic",              # labels
                   "group", "split"]                             # for splitting

df[columns_to_save].to_csv("train_data_with_preprocessing2.csv", index=False)
print("Saved clean_data.csv with", len(df), "rows")
df[columns_to_save].head()

Saved clean_data.csv with 19431 rows


,row_id,previous_reply,target_reply,prev_clean,target_clean,label,label_id,is_toxic,group,split
0,ef2a32c7af5a0a660e0c,I shared photos from today's event.,"In this video, your body draws more attention ...",i shared photos from today event,in this video your body draws more attention t...,Sexual Harassment / Objectification,9,1,0,train
1,c7a506deea76fe585973,I see no reason to change my opinion.,"Hearing this statement, it seems imagination d...",i see no reason to change my opinion,hearing this statement it seems imagination di...,Mockery / Ridicule,7,1,1,train
2,5ff6cb853fa2d2809023,"Someone in the discussion said, There is no ne...",Not everyone is the same; people should be tre...,there is no need to take men seriously in this...,not everyone is the same people should be trea...,None,0,0,2,train
3,577f23b0dbafdca5c219,What's the update on today's work?,"So far, More details can be given later.",what is the update on today work ?,so far more details can be given later,None,0,0,3,train
4,ca4f30bb8940a17405a0,I only disagreed politely.,"I'm warning you, if you ask this question agai...",i only disagreed politely,i am warning you if you ask this question agai...,Threat,10,1,4,train


## Done!

The file `clean_data.csv` is ready. In the next notebook you will:
```python
data  = pd.read_csv("clean_data.csv")
train = data[data.split == "train"]
val   = data[data.split == "val"]
test  = data[data.split == "test"]      # touch only at the very end
```